In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-1: Post-training: the landscape

## Where this session fits

> **The limitation we start from: the intro series showed one RL step on a language model with a reward it was simply given. It never said how a pretrained model becomes an assistant, or where the reward comes from.**

- **The question:** how does a pretrained model become an assistant, and where do SFT, RL and preferences fit?
- **Position: session 1 of the RL for LLMs series (RLLM-1 to RLLM-13)**
    - a **session** is one deck of the series
    - it builds on the RL intro series (Parts A-C)
    - the next session is RLLM-2: preferences, reward models and RLHF
- **Series goal:** turn a pretrained language model into a useful, aligned assistant with reinforcement learning, and know the methods well enough to choose among them
    - this session gives the map and a first rule for choosing (our reading)

**By the end you can**

1. say what post-training adds to a pretrained model, and name its common stages
2. say what supervised fine-tuning and reinforcement fine-tuning each consume
3. name the sources of a reward, and say who supplies the comparisons behind a reward model
4. say which session of the series covers which method, and pick a first method from what you have

**What we reuse from the intro, in six lines**

- **Parts A and B:** a policy chooses actions
    - policy gradient raises the probability of actions with a positive **advantage** (return minus a baseline)
- **PPO** clips the probability ratio, so a batch can be reused
    - a **critic** learns the baseline
- **intro session C1 (Language models):** a language model is a policy over tokens
    - state: the prompt and the tokens so far
    - a **completion** is the model's whole output for a prompt: one episode
- **C1:** PPO adds a KL penalty to a **reference model**, a frozen copy of the starting model
    - the reward is only a **proxy**, so the policy can exploit it
- **intro session C2 (GRPO):** GRPO drops the critic: a group of completions for one prompt supplies the baseline
- if a term is hazy, the [Review and Preview: Reinforcement Learning](../../Review_Advanced_Reinforcement_Learning.ipynb) notebook links the intro sessions


# A pretrained model is not yet an assistant

**Why this section?** Everything in this series starts from a model that was trained only to predict the next token.

> **Pretraining teaches a model to continue text. Post-training teaches it to do what people ask.**

- **pretraining:** predict the next token of a huge text collection
    - supervised learning on pairs (a prefix, the next token)
    - SFT will use the same loss on better pairs
- the user wants a question answered, not a text continued
    - the pretrained model continues: "The Black-Scholes equation states ..."
    - the user asks: "What does the Black-Scholes equation state?"
- **post-training** turns the pretrained model into an assistant
    - it is a sequence of **stages**, each one training phase
    - not an *RL step*, one update, as in the intro
    - each stage may add new knowledge or new behavior


## Two ways to teach: show how, or say what counts

**Why this section?** Every post-training stage uses one of two kinds of feedback. The intro series introduced both in its session A0 (Tour).

> **Supervised fine-tuning shows the model what to say. Reinforcement fine-tuning rates what it said.**

| | supervised fine-tuning (**SFT**) | reinforcement fine-tuning (**RFT**) |
|:--|:--|:--|
| the update consumes | a **demonstration**: a prompt with a good completion written for it | a reward for the model's own completion |
| the model learns | to imitate | to meet a criterion |
| needs | demonstrations | a reward, and a model that sometimes succeeds |
| risk (our reading) | copies the flaws | exploits the reward |

- **RFT**: PPO and GRPO (Parts B and C) are the standard ways
    - RLLM-7 to 13 compare other ways

**Why SFT comes first**

> **RFT can only raise the probability of what the model already does sometimes (our reading of intro session C1). A pretrained model that never writes a good completion gives RFT nothing to reinforce.**

- so a common recipe starts with SFT: it makes good completions likely enough to be found
- RFT then improves on them, using rewards instead of demonstrations
- an exception: RLLM-4 applies RL straight to a pretrained model, when a verifier makes good completions findable


## The common post-training steps

**Why this section?** The stages differ in what they teach and in the feedback they use.

> **Each stage mostly teaches one thing, with either a demonstration (SFT: the right next token at every position) or a reward (RL).**

| Stage | What it teaches | Feedback |
|:--|:--|:--|
| instruction tuning | follow written instructions and chat in turns | demonstrations (SFT) |
| alignment (behaving as people want) | helpful, honest, harmless answers | comparisons of two completions, then RL (**RLHF**, defined below) |
| tool use | when and how to call an outside tool (search, calculator) | demonstrations or RL |
| reasoning | think step by step before answering | **verifiable rewards**: a program checks the final answer (RL) |
| others (domain adaptation, safety) | specialist knowledge; refusing harmful requests | demonstrations or RL |

- the stages are often repeated or combined
- the table is a menu, not a fixed order

**One common recipe**

<img src="images/rllm1_pipeline.png" width=95%>

- the preference stage and reasoning RL differ in where the reward comes from (next slide)
- **distillation:** train a smaller model on the outputs of this one
- **RLHF and kin:** RLHF and similar methods that learn from comparisons (RLLM-2, RLLM-3)


## Where the reward comes from

**Why this section?** RL needs a reward, and a student of this series must know who or what supplies it.

> **A reward comes from a program that checks the answer, or from a reward model trained on comparisons. A prompted judge model can supply those comparisons, or rate a completion directly.**

| Source | What gives the reward | Session |
|:--|:--|:--|
| **verifier** | a program checks the final answer | RLLM-4 |
| **reward model** | a network trained on comparisons of completions | RLLM-2 |
| **model as judge** | a prompted model supplies the labels (comparisons) that train a reward model | RLLM-3 |

- a **comparison** says which of two completions is better
- a **preference** is one such comparison

**Who supplies the comparisons**

1. **RLHF**: reinforcement learning from human feedback; people make the comparisons (RLLM-2)
2. **RLAIF**: reinforcement learning from AI feedback; a prompted model does (RLLM-3)
3. **Constitutional AI**: a model critiques and revises its answers against written principles, and the comparisons follow from them (RLLM-3)

- in all three, the comparisons train a reward model
    - with a judge, the judge model produces the labels; the reward model is still learned
- only the source of the comparisons changes


## What you have decides the first method to try

**Why this section?** The series goal is to choose among methods. This is a first rule for choosing, built from this session alone.

<img src="images/rllm1_choose.png" width=80%>

- our reading: a starting point
- cost, risk and data quality change the answer
- **DPO** (direct preference optimization) learns from comparisons without a separate reward model (RLLM-10)
- **distillation** here: a stronger model's outputs are the demonstrations (RLLM-11)
- RLLM-13 returns to choosing, with all the methods on one map


# The map of the series

**Why this section?** Every later session adds one thing to this map. The same picture opens RLLM-2 to RLLM-6, with that session marked.

<img src="images/series_map_1.png" width=85%>

- sessions 2 to 6 build the core recipes
- sessions 7 to 13 are part 2 of the RLLM series: RFT Alternative methods

**Why three questions: PPO and GRPO are not the only choice**

> **Any method that trains a model from feedback answers three questions. RLLM-7 to 13 vary one answer at a time, so we preview the questions here.**

<img src="images/rllm1_three_questions.png" width=68%>

- **on-policy:** the update uses completions sampled from the model as it is now
- **off-policy:** it uses older completions, or another model's
- a **signal** is the feedback number the update uses
    - its grain: one per completion, or one per token
    - PPO's reward arrives once per completion; its critic spreads credit to each token
    - GRPO gives every token the completion's advantage
- **self-training:** the model learns from its own filtered completions (RLLM-9)

**What each session adds**

| Id | Session | Adds |
|:--|:--|:--|
| RLLM-2 | Preferences, reward models and RLHF | the reward from comparisons, the full RLHF pipeline |
| RLLM-3 | RLAIF and Constitutional AI | a model replaces the human labeler |
| RLLM-4 | Case study: training a reasoning model | verifiable rewards, end to end |
| RLLM-5 | PPO in depth | what PPO needs in practice, and where the clip comes from |
| RLLM-6 | GRPO and its refinements | known weak spots and their fixes |
| RLLM-7 to 13 | RFT Alternative methods (part 2 of the RLLM series) | the three questions varied one at a time: DPO, self-training, on-policy distillation |


# What we established, and what it sets up

- **Established**
    - post-training turns a model that continues text into an assistant, in steps
    - SFT shows what to say; RFT rates what was said
    - SFT usually comes first (our reading); RLLM-4 shows a case that skips it
    - a reward comes from a verifier or a reward model
    - the comparisons behind a reward model come from people or from a judge model
    - a first rule for choosing, and the map of RLLM-2 to 13
- **The limit of this session:** it is a map
    - it runs no method; the choosing rule is only a first pass
- **Sets up**
    - **the limitation:** how does a pile of comparisons become a reward?
    - next session, RLLM-2: preferences, the Bradley-Terry model, reward models, and the RLHF pipeline
